# One Small Change, a Completely Different Hash
A good hash function changes its whole output when the input changes even slightly. This is called the **avalanche effect**.

## Step 1: Change one character
Each message below differs from "hello" by only one character.

In [1]:
import hashlib

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

for message in ["hello", "Hello", "hello.", "hellp"]:
    print(f"{message!r:9} {sha256_text(message)}")

'hello'   2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
'Hello'   185f8db32271fe25f561a6fc938b2e264306ec304eda518007d1764826381969
'hello.'  1589999b0ca6ef8814283026a9f166d51c70a910671c3d44049755f07f2eb910
'hellp'   fdd7585e08c4e2afd71dcabdb4636c89d557a3f42db9e2040c8bbd1708aa4ce7


## Step 2: Count how many bits changed
"hello" and "Hello" differ by only **1 bit** in the input. We turn both hashes into bits and count how many positions are different.

In [2]:
def to_bits(data: bytes) -> str:
    # each byte becomes 8 bits, using the 08b code from 01-bits-bytes-hex
    return "".join(format(byte, "08b") for byte in data)

# The inputs: h and H
print("h:", format(ord("h"), "08b"))
print("H:", format(ord("H"), "08b"))

# The outputs
a = to_bits(hashlib.sha256(b"hello").digest())
b = to_bits(hashlib.sha256(b"Hello").digest())

changed = sum(1 for x, y in zip(a, b) if x != y)
print(f"{changed} of {len(a)} bits changed")

h: 01101000
H: 01001000
125 of 256 bits changed


## Step 3: Test many messages
One result could be luck. Here we compare the hash of "hello" with the hashes of 1,000 similar messages ("hello0", "hello1", ...) and look at the average number of bits that change.

In [3]:
base_bits = to_bits(hashlib.sha256(b"hello").digest())

results = []
for i in range(1000):
    other_bits = to_bits(hashlib.sha256(f"hello{i}".encode("utf-8")).digest())
    changed = sum(1 for x, y in zip(base_bits, other_bits) if x != y)
    results.append(changed)

print("Average bits changed:", sum(results) / len(results))
print("Fewest:", min(results))
print("Most:", max(results))

Average bits changed: 127.895
Fewest: 106
Most: 160


## Summary
- Changing even one bit of the input changes about half of the 256 output bits (125 in my test of "hello" vs "Hello").
- Over many messages, the average is very close to 128, which is half.
- Half is the ideal: the two hashes look completely unrelated, so the hash gives away nothing about how similar two messages are.
- No part of the old hash survives in the new one, so you can't "partly match" a hash. It either matches exactly or it doesn't.